# Tokenization

**INPUT**: Preprocessed train, validation, test corpora

**OUTPUT**: LSTM and LLM tokenizer model & vocabulary

| Step | Decision | Status | Comment |
|------|----------|--------|---------|
| Train tokenizer | - | Revision | Parameter decision, missing after preprocessing eda and after tokenization eda |
| LSTM tokenization | - | - | - |

## Input & Setup

### Imports

In [ ]:
import pandas as pd
import sentencepiece as spm
from config import PATHS
import pickle

### Read corpora

In [ ]:
train = pd.read_csv(PATHS.train_processed)
validation = pd.read_csv(PATHS.validation_processed)
test = pd.read_csv(PATHS.test_processed)

### Prepare tokenizer training data

In [ ]:
train["Message"].to_csv(PATHS.lstm_tokenizer_train_data, index=False, header=False)

## Steps

### Train tokenizer

In [ ]:
tokenizer_model_prefix = str(PATHS.lstm_tokenizer.with_suffix(''))
spm.SentencePieceTrainer.train(
    input=str(PATHS.lstm_tokenizer_train_data),
    model_prefix=tokenizer_model_prefix,
    vocab_size=30000,
    user_defined_symbols=["<MAIL>", "<URL>", "<NUM>", "<PHONE>"],
    model_type='unigram',
    character_coverage=1.0
)


### Tokenization test

In [ ]:
first_message = train["Message"].iloc[0]
print(first_message)

sp = spm.SentencePieceProcessor()
sp.load(str(PATHS.lstm_tokenizer))

ids = sp.encode(first_message)
print(ids)
text = sp.decode(ids)
print(text)
ids = sp.encode(text, out_type=int)
pieces = sp.encode(text, out_type=str)

for token_id, subword in zip(ids, pieces):
    print(f"ID {token_id:5d} -> '{subword}'")

### Tokenize

In [ ]:
def encode_messages(messages):
    token_ids = [sp.encode(msg, out_type=int) for msg in messages]
    token_pieces = [sp.encode(msg, out_type=str) for msg in messages]
    return token_ids, token_pieces

train_ids, train_pieces = encode_messages(train["Message"])
validation_ids, validation_pieces = encode_messages(validation["Message"])
test_ids, test_pieces = encode_messages(test["Message"])

for name, id, piece, id_pkl, piece_pkl in zip(
    ["train", "validation", "test"],
    [train_ids, validation_ids, test_ids],
    [train_pieces, validation_pieces, test_pieces],
    [PATHS.train_ids, PATHS.validation_ids, PATHS.test_ids],
    [PATHS.train_pieces, PATHS.validation_pieces, PATHS.test_pieces]
):
    with open(f"{PATHS.models}/{name}_ids.pkl", 'wb') as f:
        pickle.dump(id, f)
    with open(f"{PATHS.models}/{name}_pieces.pkl", 'wb') as f:
        pickle.dump(piece, f)